# Consumer Behavior Decision Intelligence Lab (CBDIL)
## Initial Exploratory and Confirmatory Analysis

This notebook conducts systematic exploratory data analysis (EDA) and confirmatory hypothesis checks on the UCI Online Retail II dataset across the five behavioral dimensions:
1. **Value**: Cumulative revenue, AOV, spend concentration
2. **Activity**: Recency intervals, order frequency, active months
3. **Breadth**: Distinct product count, basket line depth
4. **Stability**: Interpurchase gap variation, cadence consistency
5. **Momentum**: Recent 90-day vs prior 90-day trajectory

Every visualization is paired with the specific analytical question it answers.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go

# Load reporting datasets
data_dir = Path('../outputs/exports')
customer_df = pd.read_csv(data_dir / 'customer_summary.csv')
segment_df = pd.read_csv(data_dir / 'segment_summary.csv')
monthly_df = pd.read_csv(data_dir / 'monthly_summary.csv')
signals_df = pd.read_csv(data_dir / 'decision_signals.csv')
controls_df = pd.read_csv(data_dir / 'data_controls.csv')

print(f"Total Customers: {len(customer_df):,}")
print(f"Total Decision Signals Triggered: {len(signals_df):,}")

### Question 1: How concentrated is total customer purchasing value?
**Objective**: Quantify whether gross spend adheres to a Pareto distribution.

In [ ]:
sorted_spend = customer_df['total_value'].sort_values(ascending=False).values
total_spend = sorted_spend.sum()

pcts = [0.01, 0.05, 0.10, 0.20]
concentration = []
for p in pcts:
    cnt = max(1, int(len(sorted_spend) * p))
    val = sorted_spend[:cnt].sum()
    share = val / total_spend
    concentration.append({'Tier': f'Top {int(p*100)}%', 'Customer Count': cnt, 'Spend Share': f"{share:.1%}"})

pd.DataFrame(concentration)

### Question 2: Are behavioral segments materially distinct across dimensions?
**Objective**: Confirm separation in median recency, order frequency, and spend.

In [ ]:
segment_df[['segment_name', 'customer_count', 'customer_share', 'total_value', 'value_share', 'median_recency', 'median_frequency', 'average_order_value']]

### Question 3: What does the month-over-month state transition distribution look like?
**Objective**: Measure stability vs churn across customer lifecycle states.

In [ ]:
trans_df = pd.read_csv(data_dir / 'state_transitions.csv')
matrix = trans_df.groupby(['previous_state', 'current_state'])['customer_count'].sum().unstack(fill_value=0)
prob_matrix = matrix.div(matrix.sum(axis=1), axis=0).round(4)
prob_matrix

### Question 4: Confirmatory Check - Do softening accounts exhibit actual order drop?
**Objective**: Verify that customers flagged as Softening show statistically significant negative momentum.

In [ ]:
softening = customer_df[customer_df['behavioral_state'] == 'SOFTENING']
engaged = customer_df[customer_df['behavioral_state'] == 'ENGAGED']

print("Softening Median Value Momentum:", softening['value_momentum_pct'].median())
print("Engaged Median Value Momentum:", engaged['value_momentum_pct'].median())